In [0]:
import requests

regions = spark.table("workspace.petropulse_ref.operating_regions").collect()

URL = "https://api.open-meteo.com/v1/forecast"
rows = []
for r in regions:
    params = {
        "latitude": r.latitude,
        "longitude": r.longitude,
        "current": "temperature_2m,wind_speed_10m,wind_gusts_10m,precipitation",
        "timezone": "UTC",
    }
    resp = requests.get(URL, params=params, timeout=30)
    resp.raise_for_status()
    cur = resp.json()["current"]
    rows.append((
        r.region_id, cur["time"],
        float(cur["temperature_2m"]), float(cur["wind_speed_10m"]),
        float(cur["wind_gusts_10m"]), float(cur["precipitation"]),
    ))
    print(r.region_name, "->", cur["wind_speed_10m"], "km/h wind,", cur["precipitation"], "mm precip")

In [0]:
from pyspark.sql import functions as F

schema = """region_id string, observed_at string, temperature_c double,
            wind_speed_kmh double, wind_gust_kmh double, precipitation_mm double"""

df = (spark.createDataFrame(rows, schema)
      .withColumn("observed_at", F.to_timestamp("observed_at"))
      .withColumn("ingested_at", F.current_timestamp()))

df.write.mode("append").saveAsTable("workspace.petropulse_bronze.openmeteo_weather_raw")
print("Rows written:", df.count())

In [0]:
%sql
SELECT region_id, observed_at, temperature_c, wind_speed_kmh, wind_gust_kmh, precipitation_mm
FROM workspace.petropulse_bronze.openmeteo_weather_raw
ORDER BY region_id;